# Attention Gradient Analysis

Gradient-based attention analysis: attention gradient attribution,
gradient-weighted patterns, sensitivity maps, head ranking, and
attention gradient flow.

## Why This Matters

Attention gradient reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_gradient_analysis import (
    attention_gradient_attribution,
    gradient_weighted_pattern,
    attention_sensitivity_map,
    gradient_head_ranking,
    attention_gradient_flow,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])

def metric_fn(logits, tokens):
    return jnp.mean(logits[-1])

print("Model ready")

## Attention Gradient Attribution

In [ ]:
result = attention_gradient_attribution(model, tokens, metric_fn, layer=0, head=0)
print(f"Gradient magnitude: {result['gradient_magnitude']:.6f}")
print(f"Top entries (query_pos, key_pos, gradient):")
for i, j, g in result['top_entries'][:5]:
    print(f"  ({i}, {j}): {g:.6f}")

## Gradient-Weighted Pattern

In [ ]:
result = gradient_weighted_pattern(model, tokens, metric_fn, layer=0)
print(f"Head importances: {[f'{float(i):.6f}' for i in result['head_importances']]}")
print(f"Head weights: {[f'{float(w):.3f}' for w in result['head_weights']]}")

## Attention Sensitivity Map

In [ ]:
result = attention_sensitivity_map(model, tokens, metric_fn, layer=0, head=0)
print(f"Position sensitivity:")
for pos, s in enumerate(result['position_sensitivity']):
    bar = '#' * int(float(s) * 1000)
    print(f"  pos {pos}: {float(s):.6f} {bar}")
print(f"Most sensitive: pos {result['most_sensitive_position']}")

## Gradient Head Ranking

In [ ]:
result = gradient_head_ranking(model, tokens, metric_fn)
print("Top 5 heads:")
for l, h, imp in result['top_heads']:
    print(f"  L{l}H{h}: importance={imp:.6f}")
print(f"\nBottom 5 heads:")
for l, h, imp in result['bottom_heads']:
    print(f"  L{l}H{h}: importance={imp:.6f}")

## Attention Gradient Flow

In [ ]:
result = attention_gradient_flow(model, tokens, metric_fn)
for l in range(cfg.n_layers):
    attn = float(result['attn_importance'][l])
    mlp = float(result['mlp_importance'][l])
    print(f"Layer {l}: attn={attn:.6f}, mlp={mlp:.6f}")
print(f"\nPeak layer: {result['peak_layer']}")
print(f"Flow direction: {'increasing' if result['flow_direction'] > 0 else 'decreasing'}")